# Cheminformatics Tutorial - Data Acquisition
**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/01_pubchem_data_retrieval.ipynb)


---


### Contents

This notebook introduces the first stage of the computational workflow: retrieving molecular data from PubChem using its REST API.

The activities are structured to guide you through the complete process of:

1. Understanding the PubChem data model  
2. Querying compounds programmatically  
3. Retrieving molecular properties  
4. Constructing a structured dataset  
5. Performing initial data inspection  
6. Exporting a raw dataset for downstream curation  

This notebook is designed to be completed in approximately **45–60 minutes**.  

You are encouraged to modify queries, explore additional properties, and inspect returned data structures. The goal is not only to retrieve molecules, but to understand how chemical information is organized and accessed computationally.

---

### PubChem Data Retrieval  

**PubChem** is a public chemical information resource maintained by the NIH, providing access to molecular structures, identifiers, and associated chemical properties.

In this notebook, we will **search PubChem**, **retrieve molecular properties programmatically**, and **construct a structured dataset** suitable for downstream curation and analysis.

---

Imports

In [ ]:
import os
import time
import requests
import pandas as pd

### Step 1: Defining the Chemical Search Space
In this section, we define the search parameters. In a real-world project, this represents the **initial library selection**. We utilize a keyword-based search to identify a specific chemical class (e.g., flavonoids) and limit the output to ensure computational efficiency during this tutorial.

In [ ]:
BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

QUERY = "flavonoid"   # cambia si quieres
MAX_CIDS = 200        # tamaño controlado para el tutorial

### Step 2: From Natural Language to Unique CIDs
Chemical names are often ambiguous or redundant. To ensure data integrity, we must convert our query into **PubChem CIDs**. The CID is a permanent, unique numerical index that acts as the "Social Security Number" for a molecule within the NIH ecosystem.

In [ ]:
def get_cids_by_name(query: str, max_cids: int = 200):
    # "name" search can return a single compound for exact names; for broad queries,
    # a safer approach is using "compound/name/<query>/cids" but results vary.
    url = f"{BASE_URL}/compound/name/{query}/cids/JSON"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return cids[:max_cids]

cids = get_cids_by_name(QUERY, MAX_CIDS)
len(cids), cids[:10]

### Step 3: Molecular Digitalization
Now, we "pull" specific data for our CIDs. We focus on properties critical for **Drug-Likeness** and **QSAR** modeling:
* **Canonical SMILES:** A string representation of the molecular graph, essential for computer "vision."
* **XLogP:** An estimate of lipophilicity, predicting how easily a drug crosses lipid membranes.
* **TPSA (Topological Polar Surface Area):** A key indicator of oral bioavailability and blood-brain barrier permeability.

In [ ]:
def fetch_properties(cids):
    props = [
        "CID",
        "CanonicalSMILES",
        "IsomericSMILES",
        "InChI",
        "InChIKey",
        "MolecularWeight",
        "XLogP",
        "TPSA",
    ]
    cid_str = ",".join(map(str, cids))
    url = f"{BASE_URL}/compound/cid/{cid_str}/property/{','.join(props)}/JSON"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()
    rows = data.get("PropertyTable", {}).get("Properties", [])
    return pd.DataFrame(rows)

df = fetch_properties(cids)
df.shape

### Step 4: Initial Data Inspection & Quality Control (QC)
"Garbage In, Garbage Out." Before moving to modeling, we must perform a sanity check on the retrieved data. This involves identifying missing values (NaNs) in critical columns and checking for structural duplicates. Even high-authority databases contain "noisy" data that must be identified early.

In [ ]:
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df.duplicated(subset=["CanonicalSMILES"]).sum()

Save data raw CSV

In [ ]:
out_path = os.path.join(OUT_DIR, "pubchem_raw.csv")
df.to_csv(out_path, index=False)
out_path

### Summary and Next Steps
You have successfully built a raw chemical dataset directly from the cloud. 
* **Current State:** We have a `.csv` file with 1D/2D properties.
* **Next Module:** We will use **RDKit** to perform "Chemical Curation," where we strip salts, handle tautomers, and prepare these molecules for 3D descriptor calculation.

**Suggested Exercise:** Change the `QUERY` variable to "alkaloid" or "kinase inhibitor" and re-run the notebook. Observe how the distribution of Molecular Weight and XLogP changes between chemical classes.